# Mini Project: Sentiment Assistant with BERT Fine-Tuning
## Week 7 — Day 5 | DI GenAI & Machine Learning Bootcamp 2026

**Scenario:** The support analytics team needs a reliable sentiment signal for long-form customer feedback so they can escalate angry customers before churn happens.

**Pipeline:**
1. Hardware check — confirm GPU
2. Load IMDB reviews dataset via TFDS
3. Tokenize with BERT WordPiece tokenizer and build a tf.data pipeline
4. Fine-tune `TFBertForSequenceClassification` (bert-base-uncased)
5. Evaluate on the held-out test set
6. Build a reusable `predict_sentiment` inference helper
7. Reflection — business implications and next steps

> **Run on Google Colab with GPU runtime** — ~15 min on T4.  
> CPU-only will work but takes significantly longer.

In [ ]:
pip install -q tensorflow tensorflow-datasets transformers accelerate evaluate

## Section 1 — Imports & Hardware Check

If `GPU devices detected: []` appears below, switch the Colab runtime to **Runtime → Change runtime type → T4 GPU** before continuing.

In [ ]:
import platform
import warnings
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification
import evaluate
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices("GPU"))

# Mixed precision for faster training on modern GPUs
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision enabled (float16) ✓")
else:
    print("No GPU detected — training on CPU (slower)")

## Section 2 — Load the IMDB Reviews Dataset

IMDB is perfectly balanced: **25,000 positive + 25,000 negative** training reviews, with an identical held-out test split. `as_supervised=True` yields `(text, label)` pairs — exactly what our model expects.

Labels: `0 = Negative`, `1 = Positive`

In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True,
)
print(ds_info)

print(f"\nTrain examples : {ds_info.splits['train'].num_examples:,}")
print(f"Test  examples : {ds_info.splits['test'].num_examples:,}")

# Quick peek at raw samples
print("\n── Sample reviews ──")
for text, label in ds_train.take(2):
    sentiment = "Positive" if label.numpy() else "Negative"
    print(f"\nLabel : {sentiment}")
    print(text.numpy().decode()[:250], "…")

## Section 3 — Tokenizer Setup & Data Pipeline

### Why these choices?

| Setting | Value | Reason |
|---|---|---|
| `MAX_LENGTH` | 256 | Covers most reviews; 512 doubles VRAM usage |
| `BATCH_SIZE` | 16 | Fits a T4 GPU with BERT; reduce to 8 on smaller GPUs |
| `do_lower_case=True` | ✓ | Must match `bert-base-uncased` vocabulary |
| `add_special_tokens=True` | ✓ | Prepends `[CLS]`, appends `[SEP]` |
| `padding="max_length"` | ✓ | Uniform shape required for batching |
| `return_attention_mask=True` | ✓ | Tells BERT which tokens are real vs pad |

`tf.py_function` bridges Hugging Face Python tokenization logic into a TensorFlow graph pipeline — no manual NumPy gymnastics required.

In [ ]:
MAX_LENGTH = 256
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print(f"Tokenizer loaded : {tokenizer.name_or_path}")
print(f"Vocab size       : {tokenizer.vocab_size:,}")
print(f"Special tokens   : [CLS]={tokenizer.cls_token_id}  "
      f"[SEP]={tokenizer.sep_token_id}  [PAD]={tokenizer.pad_token_id}")


def encode_review(review_input):
    """Convert raw bytes / string / tensor to BERT input dict."""
    if isinstance(review_input, bytes):
        review_text = review_input.decode("utf-8")
    elif hasattr(review_input, "numpy"):
        review_text = review_input.numpy().decode("utf-8")
    else:
        review_text = str(review_input)

    return tokenizer.encode_plus(
        review_text,
        add_special_tokens   = True,
        max_length           = MAX_LENGTH,
        padding              = "max_length",
        truncation           = True,
        return_attention_mask    = True,
        return_token_type_ids   = True,
    )


def tf_encode(text, label):
    """Wrap Python tokenizer in a tf.py_function so it runs inside tf.data."""
    encoded = tf.py_function(
        func = lambda t: list(encode_review(t).values()),
        inp  = [text],
        Tout = [tf.int32, tf.int32, tf.int32],
    )
    return {
        "input_ids"      : encoded[0],
        "attention_mask" : encoded[1],
        "token_type_ids" : encoded[2],
    }, label


def prepare_dataset(dataset):
    return (
        dataset
        .map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
        .shuffle(2000)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )


train_ds = prepare_dataset(ds_train)
test_ds  = prepare_dataset(ds_test)

print(f"\nPipeline built ✓")
print(f"  Train batches : {len(train_ds)}")
print(f"  Test  batches : {len(test_ds)}")

# Inspect one batch
for batch, labels in train_ds.take(1):
    print(f"\nBatch keys     : {list(batch.keys())}")
    print(f"input_ids  shape : {batch['input_ids'].shape}")
    print(f"attn_mask  shape : {batch['attention_mask'].shape}")
    print(f"labels     shape : {labels.shape}")

## Section 4 — Initialize and Fine-Tune `TFBertForSequenceClassification`

### Why these hyperparameters?

| Hyperparameter | Value | Rationale |
|---|---|---|
| `learning_rate` | `2e-5` | Standard BERT fine-tuning range (1e-5 to 5e-5); larger values destabilize pretrained weights |
| `epsilon` | `1e-8` | Adam stability for float16 arithmetic |
| `loss` | `SparseCategoricalCrossentropy(from_logits=True)` | BERT outputs raw logits — softmax applied inside the loss |
| `EPOCHS` | `2` | Sufficient for IMDB; >3 tends to overfit without LR scheduling |

`use_safetensors=False` avoids a compatibility issue in older transformers versions.

In [ ]:
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels       = 2,
    use_safetensors  = False,
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn   = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics   = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()

### Training

Watch the **validation accuracy** plateau — that is the natural stopping signal. Two epochs should comfortably hit **≥ 90%** on a T4 GPU.  
Screenshot the learning curves for your portfolio.

In [ ]:
EPOCHS = 2   # increase to 3 if time allows and GPU is available

history = model.fit(
    train_ds,
    validation_data = test_ds,
    epochs          = EPOCHS,
)

In [ ]:
# ── Learning curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_r = range(1, EPOCHS + 1)

axes[0].plot(epochs_r, history.history["loss"],          "b-o", label="Train")
axes[0].plot(epochs_r, history.history["val_loss"],      "r-o", label="Val")
axes[0].set_title("Loss", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_r, history.history["accuracy"],      "b-o", label="Train")
axes[1].plot(epochs_r, history.history["val_accuracy"],  "r-o", label="Val")
axes[1].set_title("Accuracy", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[1].axhline(0.90, color="green", linestyle="--", linewidth=1, label="90% target")

plt.suptitle("BERT Fine-Tuning on IMDB Reviews", fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("bert_training_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved ✓")

## Section 5 — Evaluate on the Held-Out Test Set

`model.evaluate` runs a full forward pass on the untouched test pipeline — mimicking a production QA gate.  
The classroom benchmark is **≥ 90% accuracy**.

In [ ]:
eval_metrics = model.evaluate(test_ds, return_dict=True)

print(f"\n{'─'*40}")
print(f"Test Loss     : {eval_metrics['loss']:.4f}")
print(f"Test Accuracy : {eval_metrics['accuracy']*100:.2f}%")

benchmark = 0.90
if eval_metrics["accuracy"] >= benchmark:
    print(f"✓ Benchmark met — accuracy ≥ {benchmark*100:.0f}%")
else:
    gap = (benchmark - eval_metrics["accuracy"]) * 100
    print(f"✗ Below benchmark by {gap:.1f}pp — consider a 3rd epoch or lower LR")
print(f"{'─'*40}")

In [ ]:
# ── Detailed metrics with the `evaluate` library ─────────────────────────────
# Collect predictions over the full test set for precision/recall/F1
all_preds, all_labels = [], []

for batch, labels in test_ds:
    logits     = model(batch, training=False).logits
    predictions = tf.argmax(logits, axis=-1).numpy()
    all_preds.extend(predictions.tolist())
    all_labels.extend(labels.numpy().tolist())

accuracy_metric  = evaluate.load("accuracy")
f1_metric        = evaluate.load("f1")
precision_metric = evaluate.load("precision")
recall_metric    = evaluate.load("recall")

results = {
    "accuracy"  : accuracy_metric.compute(predictions=all_preds,  references=all_labels)["accuracy"],
    "f1"        : f1_metric.compute(predictions=all_preds,        references=all_labels, average="binary")["f1"],
    "precision" : precision_metric.compute(predictions=all_preds, references=all_labels, average="binary")["precision"],
    "recall"    : recall_metric.compute(predictions=all_preds,    references=all_labels, average="binary")["recall"],
}

print("Detailed Test Metrics")
print(f"{'─'*35}")
for k, v in results.items():
    print(f"  {k:<12} : {v*100:>6.2f}%")
print(f"{'─'*35}")

## Section 6 — Reusable `predict_sentiment` Inference Helper

The helper wraps the full tokenize → forward → softmax → label pipeline so any support transcript can be scored in one call.  
The **confidence score** determines auto-response vs. human escalation routing.

In [ ]:
def predict_sentiment(text: str):
    """
    Score a single piece of text for sentiment.

    Returns
    -------
    label      : "Positive" or "Negative"
    confidence : float in [0, 1] — softmax probability of the predicted class
    """
    encoded = encode_review(text)

    # Add batch dimension and cast to tensors
    input_ids       = tf.expand_dims(tf.constant(encoded["input_ids"],       dtype=tf.int32), 0)
    attention_mask  = tf.expand_dims(tf.constant(encoded["attention_mask"],  dtype=tf.int32), 0)
    token_type_ids  = tf.expand_dims(tf.constant(encoded["token_type_ids"],  dtype=tf.int32), 0)

    logits = model(
        {"input_ids": input_ids,
         "attention_mask": attention_mask,
         "token_type_ids": token_type_ids},
        training=False,
    ).logits                                             # (1, 2)

    probs     = tf.nn.softmax(logits, axis=-1).numpy()[0]   # (2,)
    pred_idx  = int(probs.argmax())
    label     = "Positive" if pred_idx == 1 else "Negative"
    return label, float(probs.max())


# ── Test on the required sentence ────────────────────────────────────────────
custom_sentence = "The onboarding emails were confusing, but the agent fixed everything politely."
label, confidence = predict_sentiment(custom_sentence)
print(f"Prediction : {label} (confidence={confidence:.3f})")

# ── Support-scenario stress test ─────────────────────────────────────────────
support_texts = [
    "I've been trying to reach someone for 3 days. This is absolutely unacceptable!",
    "The issue was resolved instantly — your team is incredible, thank you so much!",
    "Decent service, took a bit long but the outcome was okay.",
    "Completely broken product. I demand a full refund immediately.",
    "I'm pleasantly surprised — setup was way easier than I expected.",
    "The onboarding emails were confusing, but the agent fixed everything politely.",
]

print(f"\n{'─'*72}")
print(f"{'Text':<50} {'Label':>10}  {'Confidence':>12}")
print(f"{'─'*72}")
for txt in support_texts:
    lbl, conf = predict_sentiment(txt)
    short = txt[:48] + "…" if len(txt) > 49 else txt
    flag  = " ⚠ ESCALATE" if lbl == "Negative" and conf > 0.85 else ""
    print(f"{short:<50} {lbl:>10}  {conf:>12.3f}{flag}")
print(f"{'─'*72}")

In [ ]:
# ── Confidence distribution on a sample of the test set ──────────────────────
sample_confs_pos, sample_confs_neg = [], []

for text, label in ds_test.take(200):
    _, conf = predict_sentiment(text)
    lbl_int = label.numpy()
    if lbl_int == 1:
        sample_confs_pos.append(conf)
    else:
        sample_confs_neg.append(conf)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(sample_confs_pos, bins=20, alpha=0.6, color="#5cb85c", label="Positive")
ax.hist(sample_confs_neg, bins=20, alpha=0.6, color="#d9534f", label="Negative")
ax.axvline(0.85, color="orange", linestyle="--", linewidth=1.5, label="Escalation threshold (0.85)")
ax.set_xlabel("Confidence score")
ax.set_ylabel("Count")
ax.set_title("Prediction Confidence Distribution — Test Set Sample (n=200)", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("confidence_distribution.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved ✓")

## Section 7 — Reflection & Next Steps

### Q1 — What lever most improved results?

**Pretraining** was by far the biggest lever. BERT's 110M parameters already encode grammar, world knowledge, and semantic relationships from BooksCorpus + Wikipedia. Fine-tuning on 25,000 labeled reviews for just 2 epochs applies a tiny gradient nudge on top of a model that already understands English deeply. Without pretraining, a random-init Transformer would need orders of magnitude more data and epochs to reach 90%.

Among the tunable knobs available during fine-tuning:
1. **Learning rate** — the 2e-5 sweet spot is critical; 1e-4 causes catastrophic forgetting of pretrained weights; 1e-6 barely moves the classifier head
2. **Sequence length** — 256 tokens captures the beginning of most reviews (usually the most opinionated part); 512 adds cost without meaningful accuracy gains on IMDB
3. **More epochs** — a 3rd epoch typically adds ~0.5pp accuracy on IMDB but risks overfitting on domain-specific datasets

---

### Q2 — Where would you add guardrails before deploying live?

| Layer | Guardrail | Why |
|---|---|---|
| **Input** | Length cap + HTML stripping + PII redaction | Prevent prompt injection and GDPR violations |
| **Confidence threshold** | Route predictions < 0.70 to a human queue | Model is uncertain → don't auto-act |
| **Label calibration** | Temperature scaling / Platt scaling | Raw softmax scores are overconfident |
| **Drift monitoring** | Weekly accuracy audit on freshly labeled samples | Customer language evolves; model degrades silently |
| **Audit log** | Log every prediction + input hash | Compliance and debugging for disputed escalations |
| **Adversarial tests** | Red-team with sarcasm, mixed-language, typos | BERT-base-uncased sees no casing — sarcasm is hard |

---

### Q3 — Which stakeholders benefit most?

| Stakeholder | Benefit |
|---|---|
| **Support lead** | Real-time escalation queue — angry customers surfaced before churn, not after |
| **Product manager** | Aggregated sentiment over product releases reveals which features trigger frustration |
| **Compliance officer** | Sentiment signal flags regulatory risk in financial/healthcare feedback automatically |
| **Agent** | Prioritized queue means agents handle high-stakes cases first; routine positives auto-close |
| **Executive** | NPS proxy from support tickets without needing a separate survey |

---

### Transferable Skills & Next Steps

- **Domain adaptation**: fine-tune on your company's own support emails (even 500–1000 labeled examples make a significant difference)
- **Multilingual support**: swap `bert-base-uncased` for `xlm-roberta-base` — same pipeline, 100 languages
- **LoRA-efficient adaptation** (from Day 4): freeze BERT, add LoRA adapters for each new business unit — minimal compute per task
- **Monitoring dashboard**: log `(text_hash, label, confidence, timestamp)` → Grafana/Looker for drift detection